In [18]:
#| default_exp grithopper.utils

In [10]:
#| export
import json, os, faiss, torch
from typing import Any, Dict, List, Optional

from torch.utils.data import Dataset

from gritlm import GritLM
from tqdm import tqdm

try:
    from huggingface_hub import snapshot_download
except Exception:
    snapshot_download = None
    

## Helper functions

In [5]:
#| export
def resolve_weight_file(weights_root: str) -> Optional[str]:
    """
    Resolve a pytorch_model.bin for the given weights_root.
    Mirrors the HippoRAG loading: try local dir/file, then Hugging Face snapshot.
    """
    # 1) Local directory containing pytorch_model.bin
    candidate_dir = weights_root
    if os.path.isdir(candidate_dir):
        maybe_bin = os.path.join(candidate_dir, "pytorch_model.bin")
        if os.path.exists(maybe_bin):
            return maybe_bin

    # 2) Direct .bin path
    if os.path.isfile(weights_root) and weights_root.endswith(".bin"):
        return weights_root

    # 3) Hugging Face snapshot download
    if snapshot_download is None:
        return None

    try_repo_ids = (
        [weights_root, "UKPLab/GritHopper-7B"]
        if weights_root != "UKPLab/GritHopper-7B"
        else [weights_root]
    )

    for repo_id in try_repo_ids:
        try:
            local_dir = snapshot_download(
                repo_id=repo_id, ignore_patterns=["*.md", "static/*"]
            )
            maybe_bin = os.path.join(local_dir, "pytorch_model.bin")
            if os.path.exists(maybe_bin):
                print(f"Downloaded GritHopper snapshot to {local_dir} using {repo_id}")
                return maybe_bin
        except Exception as ie:
            last_error = ie  # noqa: F841
            continue

    return None
    

In [6]:
#| export
def load_grithopper_model(weights_root: str, device: str = "cpu") -> GritLM:
    """
    Load the GritLM base model and apply GritHopper weights from HF/local,
    matching the HippoRAG loading path.
    """
    model = GritLM(
        model_name_or_path=DEFAULT_MODEL_BASE,
        mode="unified",
        pooling_method="mean",
        normalized=True,
        projection=None,
        is_inference=True,
        embed_eos="",
        attn="bbcc",
        torch_dtype=torch.bfloat16,
    )

    weight_file = resolve_weight_file(weights_root)
    if weight_file and os.path.exists(weight_file):
        try:
            state = torch.load(weight_file, map_location="cpu")
            model.load_state_dict(state, strict=False)
            print(f"Loaded GritHopper weights from {weight_file}")
        except Exception as exc:  # pragma: no cover - defensive
            print(f"Failed to load weights from {weight_file}: {exc}")
    else:
        print(f"No weight file found for {weights_root}; using base model only.")

    try:
        model.to(device)
    except Exception:
        # Some gritlm versions wrap the model attribute
        try:
            model.model.to(device)
        except Exception as exc:
            print(f"Could not move model to {device}: {exc}")

    return model


In [16]:
#| export
def generate_rows_for_dataset_data(dataset_data: Dict[str, Any]) -> List[str]:
    """
    Given the model_data dictionary, generate LaTeX table rows for each dataset.
    Returns a list of strings, each representing a LaTeX table row for a dataset.
    """
    rows = []

    # Extract hits_at_k_data and sample counts
    hits_at_k_data = dataset_data["hits_at_k"]
    n_samples = dataset_data["n_samples"]
    n_one_step = dataset_data.get("n_one_step", 0)
    n_two_step = dataset_data.get("n_two_step", 0)
    n_three_step = dataset_data.get("n_three_step", 0)
    n_four_step = dataset_data.get("n_four_step", 0)

    # Cumulative samples at each hop
    total_samples = n_one_step + n_two_step + n_three_step + n_four_step
    n_two_four_steps = n_two_step + n_three_step + n_four_step
    n_three_four_steps = n_three_step + n_four_step
    n_four_steps = n_four_step

    # Initialize row values
    row_values = []

    # List of HITs@K values, ensuring correct order
    hits_k = ["1", "5", "10"]

    # Calculate percentages for each Hits@K and hop, then add to row
    for k in hits_k:
        # Step 1 for Hits@K=k
        step1_value = (
            hits_at_k_data["step1"].get(k, 0) / total_samples * 100
            if total_samples > 0
            else "-"
        )
        row_values.append(f"{step1_value:.2f}" if step1_value != "-" else "-")

        # Step 2 for Hits@K=k
        step2_value = (
            hits_at_k_data["step2"].get(k, 0) / n_two_four_steps * 100
            if n_two_four_steps > 0
            else "-"
        )
        row_values.append(f"{step2_value:.2f}" if step2_value != "-" else "-")

        # Step 3 for Hits@K=k
        step3_value = (
            hits_at_k_data["step3"].get(k, 0) / n_three_four_steps * 100
            if n_three_four_steps > 0
            else "-"
        )
        row_values.append(f"{step3_value:.2f}" if step3_value != "-" else "-")

        # Step 4 for Hits@K=k
        step4_value = (
            hits_at_k_data["step4"].get(k, 0) / n_four_steps * 100
            if n_four_steps > 0
            else "-"
        )
        row_values.append(f"{step4_value:.2f}" if step4_value != "-" else "-")

        # Calculate weighted average for Hits@K=k
        first_acc = (
            hits_at_k_data["step1"].get(k, 0) / total_samples
            if total_samples > 0
            else 0
        )
        second_acc = (
            hits_at_k_data["step2"].get(k, 0) / n_two_four_steps
            if n_two_four_steps > 0
            else 0
        )
        third_acc = (
            hits_at_k_data["step3"].get(k, 0) / n_three_four_steps
            if n_three_four_steps > 0
            else 0
        )
        fourth_acc = (
            hits_at_k_data["step4"].get(k, 0) / n_four_steps
            if n_four_steps > 0
            else 0
        )

        weighted_avg_acc = (
            (first_acc * total_samples)
            + (second_acc * n_two_four_steps)
            + (third_acc * n_three_four_steps)
            + (fourth_acc * n_four_steps)
        ) / (
            total_samples + n_two_four_steps + n_three_four_steps + n_four_steps
        ) * 100

        # Append formatted weighted average
        row_values.append(
            "\\cellcolor{lightgray}\\itshape "
            + (f"{weighted_avg_acc:.2f}" if weighted_avg_acc != 0 else "-")
        )
    
    # Join row values with '&' and add the ending '\\' without any extra escaping
    row_str = " & ".join(row_values) + " \\\\"
    rows.append(row_str)
    
    return rows


In [8]:
#| export
def describe_hits_calc(dataset_results: Dict[str, Any], dataset_name: str) -> None:
    total_samples = (
        dataset_results.get("n_one_step", 0)
        + dataset_results.get("n_two_step", 0)
        + dataset_results.get("n_three_step", 0)
        + dataset_results.get("n_four_step", 0)
    )
    print(
        f"\nHits@1 calculation for {dataset_name}: "
        "step1 counts / all samples; "
        "step2 / samples with >=2 hops; "
        "step3 / samples with >=3 hops; "
        "step4 / samples with 4 hops."
    )
    print(f"Sample counts -> total: {total_samples}, "
          f"1-hop: {dataset_results.get('n_one_step', 0)}, "
          f"2-hop: {dataset_results.get('n_two_step', 0)}, "
          f"3-hop: {dataset_results.get('n_three_step', 0)}, "
          f"4-hop: {dataset_results.get('n_four_step', 0)}")
    

## `GritHopperEvaluator`

In [17]:
#| export
class GritHopperEvaluator:

    def get_lbl_index(all_evidence_facts_encoded:torch.Tensor):
        dimension = all_evidence_facts_encoded.shape[1]
        use_gpu = faiss.get_num_gpus() > 0 and torch.cuda.is_available()
        if use_gpu:
            res = faiss.StandardGpuResources()
            index = faiss.IndexFlatL2(dimension)
            search_index = faiss.index_cpu_to_gpu(res, 0, index)
        else:
            search_index = faiss.IndexFlatL2(dimension)
        search_index.add(all_evidence_facts_encoded.numpy())
        return search_index

    def evaluate(
        self,
        mname:str,
        dataset:Dataset,
        ks:Optional[List]=[1, 5, 10, 25, 100],
        lbl_file:Optional[str]=None,
        **kwargs
    ):
        DEFAULT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        
        tokens = mname.rstrip("/").split("/")
        if len(tokens) >= 2 and tokens[-1].startswith("tmp-checkpoint-"):
            model_name = tokens[-2]
            checkpoint_num = tokens[-1].replace("tmp-checkpoint-", "")
            model_identifier = f"{model_name}_checkpoint{checkpoint_num}"
        else:
            model_name = tokens[-1]
            checkpoint_num = ""
            model_identifier = model_name
    
        # Load model once for all datasets (HF-style, like HippoRAG)
        model = load_grithopper_model(mname, device=DEFAULT_DEVICE)

        # Initialize dataset results
        dataset_results = {
            "hits_at_k": {
                "step1": {str(k): 0 for k in ks},
                "step2": {str(k): 0 for k in ks},
                "step3": {str(k): 0 for k in ks},
                "step4": {str(k): 0 for k in ks},
            },
            "n_samples": len(dataset),
            "n_one_step": 0,
            "n_two_step": 0,
            "n_three_step": 0,
            "n_four_step": 0,
        }

        # Create a list of all unique evidence facts
        all_evidence_facts = list(dataset.data.lbl_info["input_text"])

        # Encode all evidence facts
        print(f"Encoding all evidence facts using model {mname}")

        if lbl_file is not None and os.path.exists(lbl_file):
            all_evidence_facts_encoded = torch.load(lbl_file)
        else:
            all_evidence_facts_encoded = []
            for fact in tqdm(all_evidence_facts, desc="Encoding evidences"):
                encoded_fact = model.encode(
                    fact,
                    instruction=gritlm_instruction("Represent the document: ", EMBED_BOS),
                    convert_to_tensor=True,
                    max_length=2048,
                )
                all_evidence_facts_encoded.append(encoded_fact.cpu())
                
            # Stack encoded facts into a single tensor
            all_evidence_facts_encoded = torch.stack(all_evidence_facts_encoded)
            
            torch.save(all_evidence_facts_encoded, lbl_file)

        # Build Faiss index (GPU if available, CPU otherwise)
        print(f"Building Faiss index")
        search_index = self.get_lbl_index(all_evidence_facts_encoded)

        # Evaluate each sample
        for sample in tqdm(dataset, desc=f"Evaluating samples with model {mname}"):
            n_hop_questions = len(sample["plbl2data_input_text"])

            # Update counts of n-step questions
            if n_hop_questions == 1:
                dataset_results["n_one_step"] += 1
            elif n_hop_questions == 2:
                dataset_results["n_two_step"] += 1
            elif n_hop_questions == 3:
                dataset_results["n_three_step"] += 1
            elif n_hop_questions == 4:
                dataset_results["n_four_step"] += 1

            # Initialize prompt
            query = sample["data_input_text"]
            prompt = f"Question: {query}\n{retrieve_action}"

            # Copy the list of evidence facts to manipulate during evaluation
            remaining_evidence_indices = list(range(len(all_evidence_facts)))
            evidence_encodings = all_evidence_facts_encoded.clone()

            target_evidences = set(sample["plbl2data_input_text"])
            remaining_target_evidences = target_evidences.copy()

            # Track failed k values for this question
            failed_k_values = set()

            # Evaluate each hop
            for step in range(n_hop_questions):
                step_key = f"step{step + 1}"

                # Encode the current prompt
                encoded_query = model.encode(
                    prompt,
                    instruction=gritlm_instruction(instruction, EMBED_BOS),
                    convert_to_tensor=True,
                    max_length=2048,
                ).cpu()

                # Compute cosine similarities
                similarities = torch.nn.functional.cosine_similarity(
                    encoded_query, evidence_encodings, dim=1
                )

                # Find the first positive passage in any k
                retrieved_evidence = None
                retrieved_idx = None

                # For each k, find positive passages
                for k in ks:
                    # Skip this k if we failed in a previous step
                    if k in failed_k_values:
                        continue

                    top_k = min(k, similarities.shape[0])
                    topk_values, topk_indices = torch.topk(
                        similarities, top_k, largest=True
                    )
                    retrieved_indices = topk_indices.tolist()

                    # Check which evidences are found in top k
                    current_found = set()
                    for idx in retrieved_indices:
                        fact = all_evidence_facts[remaining_evidence_indices[idx]]
                        if fact in remaining_target_evidences:
                            current_found.add(fact)
                            # Store the first positive passage found for actual retrieval
                            if retrieved_evidence is None:
                                retrieved_evidence = fact
                                retrieved_idx = idx

                    # Update successful retrievals for this k
                    if current_found:
                        dataset_results["hits_at_k"][step_key][str(k)] += 1
                    else:
                        # If nothing found for this k, mark it as failed
                        failed_k_values.add(k)

                # If no positive passage was found in any k, break
                if retrieved_evidence is None:
                    break

                # Update prompt with the retrieved evidence
                prompt += f"Document: {retrieved_evidence}\n"
                prompt += "Action: Evaluating retrieved Document: Relevant.\n"
                prompt += "Extracted information: Think yourself\n"
                prompt += retrieve_action

                # Remove the retrieved evidence from consideration
                evidence_encodings = torch.cat(
                    [
                        evidence_encodings[:retrieved_idx],
                        evidence_encodings[retrieved_idx + 1 :],
                    ],
                    dim=0,
                )
                del remaining_evidence_indices[retrieved_idx]
                remaining_target_evidences.remove(retrieved_evidence)

                # Break if all evidences have been retrieved
                if len(remaining_target_evidences) == 0:
                    break

        # Save LaTeX rows / aggregated hits including hits@1
        rows = generate_rows_for_model_data(dataset_results)
    
        return dataset_results, rows
        